# 04 — Modelling: Nightly Listing-Price Prediction

**Project:** Property Investment Intelligence System (BIU DS23)
**Owner:** Idan Gilad — models and quantitative evaluation
**Status:** Template. Runs end to end on synthetic data generated in memory (`USE_SYNTHETIC = True`) until the first real snapshot is processed. No real results.

## Purpose

Train and compare models that predict a listing's **nightly listing price** (the Inside Airbnb `price` field, local currency per night) against the charter baseline, using **grouped cross-validation inside the training partition**. Every number in this notebook is a development result, not a final claim.

## Inputs and outputs

| | Location |
| --- | --- |
| Input | `data/processed/listings_features.parquet` from `03_Feature_Engineering`: the raw listings of `01_Data_Preparation` plus engineered columns (all cities; this notebook keeps one, `CITY`). Which engineered columns may be features is decided by `evaluation/feature_audit.csv` |
| Input | Frozen split manifest `evaluation/splits/split_v1_<city>.json`, created once per city by `src/models/make_split.py` |
| Output | Model artifacts `models/<model>_<city>_<version>` and `models/cqr_<city>_<version>` (excluded from Git) |

## Guardrails

- The target is the nightly **listing price**, not acquisition value, realized revenue or occupancy.
- **One city per model** (README): set `CITY` in section 1; the currency is looked up in `src/models/cities.py`. Every file this notebook writes carries the city's name, so several cities can be modelled side by side.
- Grain is city + snapshot date + listing id. The same listing id must not appear in two partitions.
- All preprocessing and aggregations are fitted on training data only.
- **This notebook never evaluates on the test partition.** Test ids are loaded only to assert that they are excluded. Held-out evaluation, the Predictions table and segment results on test belong to `05_model_evaluation` (design, chapter 18).
- Data preparation, baselines and model definitions are imported from `src/models/`, the same code `05_model_evaluation` uses.

## 1. Setup

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json
import time

import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# TODO: pin versions in requirements.txt once the library set is agreed (team baseline: Python 3.11)
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

In [ ]:
# Paths (notebook lives in notebooks/, so the repo root is one level up)
REPO_ROOT = Path.cwd().parent
DATA_DIR = REPO_ROOT / "data" / "processed"
MODELS_DIR = REPO_ROOT / "models"
SPLITS_DIR = REPO_ROOT / "evaluation" / "splits"
FEATURE_AUDIT_FILE = REPO_ROOT / "evaluation" / "feature_audit.csv"

# Shared modelling code lives in src/models/ (design, chapter 11)
import sys
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
from src.models.make_split import (make_split, load_manifest, write_manifest, summarise,
                                   load_listings, ensure_snapshot_date, select_city)
from src.models.feature_audit import load_audit, apply_row_transforms, select_features, describe
# Shared with 05_model_evaluation, so both notebooks prepare data and build models identically
from src.models.data import (prepare_listings, make_synthetic_listings, ID_COL, SNAPSHOT_COL,
                             TARGET)
from src.models.baselines import (GlobalMedianBaseline, BASELINE_GROUPS, B1_MIN_COUNT,
                                  B1_DESCRIPTION, make_b1, make_b1_simple)
from src.models.pipelines import (make_preprocessor, make_model, COMPLEXITY_ORDER, knn_k_of,
                                  DEFAULT_G2_PARAMS, DEFAULT_KNN_K)
from src.models.target_policy import fit_target_policy, apply_target_policy
from src.models.tuning import random_search, best_params, rough_importance, SEARCH_SPACE
from src.models.intervals import fit_cqr, predict_interval, save_cqr
from src.models.cities import currency_for, city_slug, split_filename, city_data_sha256

# ---- Data source switch --------------------------------------------------
# True  -> synthetic Listings data generated in memory (src/models/data.py; no file is read)
# False -> data/processed/listings_features.parquet produced by 03_Feature_Engineering
#          (listings_master.parquet from 01_Data_Preparation also works: the engineered
#          features are then simply absent)
USE_SYNTHETIC = False
REAL_DATA_FILE = DATA_DIR / "listings_features.parquet"

# One city per model (README). The processed file holds every city in `location`.
# To model another city, change CITY and run make_split once for it (section 4).
CITY_COL = "location"
CITY = "dummy_city" if USE_SYNTHETIC else "Amsterdam"
CITY_SLUG = city_slug(CITY)          # used in every file name: "Greater Manchester" -> greater_manchester
# Prices show "$" in every city but are in the local currency (src/models/cities.py)
CURRENCY = currency_for(CITY)

# ---- Target policy (design 3.2): thresholds are learned on the training partition only ----
TARGET_POLICY_SETTINGS = {"price_upper_quantile": 0.995,   # drop placeholder prices above this
                          "max_minimum_nights": 30,        # long-term rentals: other market
                          "exclude_inactive": False}       # optional (design 3.2)

# ---- G2 (design 6, 7.1, 8): tuned gradient boosting + out-of-fold kNN price feature ----
TUNE_G2 = True        # False: G2 uses DEFAULT_G2_PARAMS (src/models/pipelines.py), no search
N_TRIALS = 30         # search budget; the design suggests 60 (each trial = 5 CV fits)


print(f"Data source: {'SYNTHETIC (in memory)' if USE_SYNTHETIC else REAL_DATA_FILE}   "
      f"city: {CITY} ({CURRENCY})")


def sha256_file(path, chunk=1 << 20):
    """Hash of the data / artifact files, recorded in the metadata and checked by `05_model_evaluation`."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()

## 2. Load the Listings table and check it against the data dictionary

The dictionary defines the Listings grain as **city + snapshot date + listing ID** and requires currency parsing and a positive price.
With synthetic data, the table is generated in memory using the dictionary's field names; no CSV is needed.
`price` may still be a raw string such as `"$1,234.00"`, and bathrooms arrive as text (`bathrooms_text`, e.g. `"1.5 shared baths"`), so both are parsed here (`01_Data_Preparation` keeps the raw Inside Airbnb values).

In [ ]:
if USE_SYNTHETIC:
    df = make_synthetic_listings(seed=RANDOM_SEED)
    DATA_SOURCE = "synthetic_in_memory"
    DATA_SHA256 = hashlib.sha256(pd.util.hash_pandas_object(df, index=False).values).hexdigest()
else:
    if not REAL_DATA_FILE.exists():
        raise FileNotFoundError(
            f"{REAL_DATA_FILE} not found. Run 01_Data_Preparation and 03_Feature_Engineering first, "
            "or set USE_SYNTHETIC = True to work with synthetic data."
        )
    df = load_listings(REAL_DATA_FILE)                # .parquet or .csv, all cities
    df = ensure_snapshot_date(df)                     # derived from scrape_id
    df = select_city(df, CITY, CITY_COL)              # one city per model
    # Fingerprint of this city's rows only: adding or reprocessing another city does not
    # make this city's models look stale. Notebook 04 and the prediction tool check it.
    DATA_SHA256 = city_data_sha256(df)
    DATA_SOURCE = REAL_DATA_FILE.name

print(df.shape)
df.head()

In [ ]:
# Quick look at the loaded table: rows per snapshot, then one line per column
print(df.groupby("snapshot_date").size().rename("rows").to_string(), "\n")
pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing_%": (df.isna().mean() * 100).round(1),
    "example": df.iloc[0].astype(str).str[:40],
})

In [ ]:
# Parsing and checks against data/data_dictionary.md -> Listings (src/models/data.py):
#   price "$1,234.00" -> 1234.0;  bathrooms / bathrooms_shared from bathrooms_text;
#   unique key (city, snapshot_date, id);  rows with missing or non-positive price removed.
# Notebook 04 calls the same function, so the test set is prepared exactly like the training data.
df = prepare_listings(df)

## 3. Target and features: the feature leakage audit (design §3.3)

The feature list is **data, not code**. `evaluation/feature_audit.csv` has one row per candidate column, classified on two axes:

- **Availability:** would this value be known for a new listing at prediction time?
- **Endogeneity:** is the value itself partly caused by the price?

An endogenous feature can produce excellent test scores while making the model useless for judging whether a price is reasonable, because the model learns the *consequence* of the price rather than its *drivers*.

Only rows with `decision = include` can become features. The other decisions are `exclude` (leakage), `ablation` (kept out of the main model, tested separately), `defer` (e.g. free text), `planned` (approved, transform not built yet), `source` (raw column used only to derive features), `identifier` and `target`. The logic lives in `src/models/feature_audit.py` so `05_model_evaluation` and the agent's prediction tool apply exactly the same list.

Checks run here:
- every `include` row that is in the data dictionary must exist in the data (core features cannot silently disappear);
- no non-`include` column can reach the feature list;
- data columns the audit does not mention are listed. With `STRICT_AUDIT = True` they stop the run. Turn it on once the audit has been completed against the real snapshot schema (design §3.3, "Watch out").

The engineered columns of `03_Feature_Engineering` are audited too. Competitor counts, amenity flags and neighbourhood counts are included. Its price-derived columns (`price_vs_neighbourhood_pct`, `neighbourhood_median_price`, `market_median_price`) are **excluded as target leakage**. Its recent-review counts are deferred, because they are counted up to a date after the snapshot.

In [ ]:
audit = load_audit(FEATURE_AUDIT_FILE)
print(f"{FEATURE_AUDIT_FILE.name}: {len(audit)} audited columns")
audit.groupby("decision")["feature"].agg(["count", lambda s: ", ".join(s)]).rename(
    columns={"<lambda_0>": "features"})

In [ ]:
STRICT_AUDIT = True    # every column of listings_features.parquet (2026-10) has a decision

df = apply_row_transforms(df, audit)       # "t"/"f" -> 1/0; row-wise, no fitting, safe before the split
empty_columns = [c for c in df.columns if df[c].isna().all()]   # e.g. instant_bookable
fs = select_features(audit, df.columns, strict=STRICT_AUDIT, empty_columns=empty_columns)

NUMERIC_FEATURES = fs.numeric                      # median-imputed
NUMERIC_INDICATOR_FEATURES = fs.numeric_indicator  # median-imputed + explicit missing flag
CATEGORICAL_FEATURES = fs.categorical
FEATURES = fs.all
used_optional = fs.not_in_dictionary   # used, but must be added to data_dictionary.md

# Segment columns for the error checks in section 8
CORE_CATEGORICAL = [c for c in ["neighbourhood_cleansed", "room_type"] if c in FEATURES]

print(f"{len(FEATURES)} features: {FEATURES}")
if fs.unaudited:
    print(f"WARNING: {len(fs.unaudited)} data columns are not in the audit and are not used: "
          f"{fs.unaudited}")
describe(fs)

In [ ]:
# Decision: model log(price) and convert back for reporting.
# exp of a log-model prediction approximates the conditional median, which suits MAE.
# TODO: record the decision and reasoning in docs/
USE_LOG_TARGET = True

def to_model_target(y):
    return np.log1p(y) if USE_LOG_TARGET else y

def from_model_target(y):
    return np.expm1(y) if USE_LOG_TARGET else y

## 4. Split manifest (frozen)

The split is **not** created here. `src/models/make_split.py` creates it once from the processed data (design, sections 4.1–4.2):

| Partition | Share of listings | Used by |
| --- | --- | --- |
| Train | ≈ 70% | this notebook: fitting, grouped CV, tuning |
| Calibration | ≈ 10% | this notebook, section 11 of the blueprint only: conformal calibration of intervals, never tuning |
| Test | ≈ 20% | `05_model_evaluation` only |

All partitions are disjoint in listing id, stratified by room type × price quintile of each listing's first snapshot. There is **one split per city**, `evaluation/splits/split_v1_<city>.json`. Create it once per city, from the repository root:

```
python -m src.models.make_split --data data/processed/listings_master.parquet --city Amsterdam
```

This notebook loads the manifest, verifies its hash and disjointness, and builds the train and calibration frames.

With synthetic data the same function is called in memory and the manifest is written to `split_synthetic.json`, so the code path is identical to the real run.

In [ ]:
if USE_SYNTHETIC:
    split_name = "split_synthetic.json"
    write_manifest(make_split(df, seed=RANDOM_SEED, id_col=ID_COL), SPLITS_DIR / split_name)
else:
    split_name = split_filename(CITY)          # e.g. split_v1_amsterdam.json
    if not (SPLITS_DIR / split_name).exists():
        raise FileNotFoundError(
            f"{SPLITS_DIR / split_name} not found. Create it once, from the repo root:\n"
            f"  python -m src.models.make_split --data data/processed/{REAL_DATA_FILE.name} --city \"{CITY}\""
        )

manifest = load_manifest(SPLITS_DIR / split_name)   # verifies hash and disjointness
SPLIT_VERSION = manifest["sha256"][:12]
if not USE_SYNTHETIC and manifest.get("city") is not None:
    assert manifest["city"] == CITY, f"{split_name} was made for {manifest['city']!r}, not {CITY!r}"
print(f"{split_name}  version {SPLIT_VERSION}  seed {manifest['seed']}  "
      f"({manifest.get('n_strata', '?')} strata)")
summarise(manifest).round(3)

In [ ]:
train_ids = set(manifest["train"])
calib_ids = set(manifest["calibration"])
test_ids = set(manifest["test"])            # loaded only to assert exclusion

# Every listing in the data must belong to the frozen split; new ids mean the data changed
unknown = set(df[ID_COL]) - (train_ids | calib_ids | test_ids)
assert not unknown, (f"{len(unknown)} listing ids are not in {split_name}. The processed data "
                     "changed after the split was frozen: investigate before re-splitting.")

train_df = df[df[ID_COL].isin(train_ids)].reset_index(drop=True)
calib_df = df[df[ID_COL].isin(calib_ids)].reset_index(drop=True)
n_test_rows = int(df[ID_COL].isin(test_ids).sum())

# Drop test rows from the working frame so nothing below can touch them by accident
df = df[~df[ID_COL].isin(test_ids)].reset_index(drop=True)
assert not test_ids & set(train_df[ID_COL]) and not test_ids & set(calib_df[ID_COL])
assert not train_ids & set(calib_df[ID_COL])

X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_calib, y_calib = calib_df[FEATURES], calib_df[TARGET]   # reserved for conformal calibration

print(f"train: {len(train_df):,} rows ({train_df[ID_COL].nunique():,} listings)   "
      f"calibration: {len(calib_df):,} rows ({calib_df[ID_COL].nunique():,} listings)   "
      f"test: {n_test_rows:,} rows excluded")

### 4.1 Target policy (design §3.2)

Which listings the model is trained and judged on. The rules are decided here, **with thresholds computed on the training partition only**, and stored in the model metadata. `05_model_evaluation` applies them unchanged to the test partition, and reports test MAE with and without the excluded rows so the filter cannot flatter the model.

| Rule | Why |
| --- | --- |
| Missing or non-positive price: excluded (already done in section 2) | no valid label |
| Price above the training partition's 99.5th percentile | placeholder prices such as 9,999 are common |
| `minimum_nights` above 30 | monthly-stay pricing is a different market |
| Inactive listings (no reviews in 12 months and fully available): **off** by default | optional in the design |

In [ ]:
target_policy = fit_target_policy(train_df, **TARGET_POLICY_SETTINGS)
removed = {}
train_df, removed["train"] = apply_target_policy(train_df, target_policy)
calib_df, removed["calibration"] = apply_target_policy(calib_df, target_policy)

X_train, y_train = train_df[FEATURES], train_df[TARGET]
X_calib, y_calib = calib_df[FEATURES], calib_df[TARGET]

print(f"price cap: {target_policy['price_upper']:,.0f} {CURRENCY}/night "
      f"({100 * target_policy['price_upper_quantile']:.1f}th percentile of training prices); "
      f"max minimum nights: {target_policy['max_minimum_nights']}")
print("The test partition is filtered by the same frozen rules in 05_model_evaluation.")
pd.DataFrame(removed)

In [ ]:
# Target distribution, training partition only, after the policy
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(y_train, bins=60)
axes[0].set(title=f"Nightly price, training partition ({CURRENCY})", xlabel=f"{CURRENCY}/night",
            ylabel="listings")
axes[1].hist(to_model_target(y_train) if "to_model_target" in globals() else np.log1p(y_train), bins=60)
axes[1].set(title="log(1 + price): the scale the models learn", xlabel="log(1 + price)")
fig.tight_layout()
plt.show()
y_train.describe().round(2)

## 5. Baseline B1: hierarchical training median (design §6.1)

The charter baseline, implemented as the design specifies. It is the median nightly listing price, computed on **training data only**, with three levels:

1. **Neighbourhood × room type**, used only when that group has at least 10 training listings.
2. Otherwise, the **room type's median** across the city.
3. Otherwise, the **global median**.

The minimum group size keeps tiny groups from producing noisy medians. A noisy baseline makes any model look better than it is.

The earlier, simpler version (neighbourhood × room type, global fallback, no minimum size) is kept as the comparison row `B1_simple`. Both live in `src/models/baselines.py`, and `05_model_evaluation` uses exactly the same definitions.

The charter's primary target is that the learned model reduces held-out MAE by **at least 10%** versus B1.

In [ ]:
# B1 = make_b1() from src/models/baselines.py (hierarchical, min group size B1_MIN_COUNT)
baseline = make_b1().fit(X_train, y_train)
print(f"B1: {B1_DESCRIPTION}")

# Which level answers the training listings (in-sample: shows how many groups are too small)
levels = pd.Series(baseline.fallback_level(X_train)).value_counts()
pd.DataFrame({"rows": levels, "share": (levels / levels.sum()).round(3)})

## 6. Learned models

Preprocessing lives inside the pipeline, so imputation, scaling and encoding are fitted on training data only.

In [ ]:
# src/models/pipelines.py builds the preprocessor from the audit's three column lists:
#   num      median impute + scale
#   num_ind  median impute + explicit was-missing flag + scale (audit transform "missing_indicator",
#            e.g. review scores, absent for new listings). The flag is added only for columns
#            with missing values in the training fold.
#   cat      most-frequent impute + one-hot, rare levels (< 10 rows) collapsed ("collapse_rare")
preprocessor = make_preprocessor(NUMERIC_FEATURES, NUMERIC_INDICATOR_FEATURES, CATEGORICAL_FEATURES)

In [ ]:
# Candidates and their settings live in src/models/pipelines.py (make_model), so `05_model_evaluation`
# can refit exactly these for its ablations and seed checks.
# TODO: add candidates (e.g. LightGBM) in make_model if the team agrees on the dependency
CANDIDATE_NAMES = ["ridge", "hist_gbm"]
models = {name: make_model(name, preprocessor, seed=RANDOM_SEED) for name in CANDIDATE_NAMES}

for name, pipe in models.items():
    pipe.fit(X_train, to_model_target(y_train))
    print(f"fitted {name}")

### 6.1 Signal checks on the training data (grouped cross-validation)

Before looking at the test set, check on the **training data only** whether the features carry real, stable signal:

- **Check A: B0 vs B1.** Does neighbourhood + room type beat a single overall median? If yes, location and room type carry information.
- **Check B: fold stability.** Every approach is scored on the same 5 grouped folds. Is the gain over B1 consistent across folds, or does it depend on one lucky split?
- **Check C: shuffled-price test.** Retrain the gradient-boosting model on randomly shuffled prices. It should do no better than B0. The gap between it and the real model is the genuine signal.

Folds are grouped by listing `id`, so all snapshots of a listing stay in the same fold. The test set is not touched here.

In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.metrics import median_absolute_error

N_FOLDS = 5


# B0, B1 (make_b1) and the comparison row B1_simple come from src/models/baselines.py

# Fixed folds, reused by every approach so fold-by-fold comparisons are fair
cv_folds = list(GroupKFold(n_splits=N_FOLDS).split(X_train, y_train, groups=train_df[ID_COL]))

# Each approach: (factory returning an unfitted estimator, trained on log target?)
CV_APPROACHES = {
    "B0_global_median": (lambda: GlobalMedianBaseline(), False),
    "B1_group_median": (lambda: make_b1(), False),           # official B1 (hierarchical)
    "B1_simple": (lambda: make_b1_simple(), False),          # earlier B1, comparison row
    **{name: (lambda p=pipe: clone(p), True) for name, pipe in models.items()},
}


def cv_evaluate(make_estimator, log_target, X, y, folds):
    '''Fold-level MAE, RMSE and MedAE in currency/night, plus out-of-fold predictions.'''
    oof = np.full(len(y), np.nan)
    scores = {"MAE": [], "RMSE": [], "MedAE": []}
    for tr, va in folds:
        est = make_estimator()
        y_fit = to_model_target(y.iloc[tr]) if log_target else y.iloc[tr]
        est.fit(X.iloc[tr], y_fit)
        pred = est.predict(X.iloc[va])
        if log_target:
            pred = from_model_target(pred)
        oof[va] = pred
        y_va = y.iloc[va]
        scores["MAE"].append(mean_absolute_error(y_va, pred))
        scores["RMSE"].append(np.sqrt(mean_squared_error(y_va, pred)))
        scores["MedAE"].append(median_absolute_error(y_va, pred))
    return {k: np.array(v) for k, v in scores.items()}, oof


def cv_mae(make_estimator, log_target, X, y, folds):
    return cv_evaluate(make_estimator, log_target, X, y, folds)[0]["MAE"]


cv_scores, cv_oof = {}, {}
for name, (make, log) in CV_APPROACHES.items():
    cv_scores[name], cv_oof[name] = cv_evaluate(make, log, X_train, y_train, cv_folds)
cv_results = {name: s["MAE"] for name, s in cv_scores.items()}   # MAE per fold, used below

cv_table = pd.DataFrame(cv_results, index=[f"fold {i + 1}" for i in range(N_FOLDS)]).T
cv_table["mean"] = cv_table.mean(axis=1)
cv_table["std"] = cv_table.iloc[:, :N_FOLDS].std(axis=1)
print(f"Grouped {N_FOLDS}-fold CV MAE ({CURRENCY}/night), training partition only")
cv_table.round(2)

#### Check A: does neighbourhood + room type carry signal? (B0 vs B1)

In [ ]:
b0, b1 = cv_results["B0_global_median"], cv_results["B1_group_median"]
gain_b1 = 100 * (1 - b1 / b0)

print(f"B1 reduces MAE vs B0 by {gain_b1.mean():.1f}% on average "
      f"(per fold: {', '.join(f'{g:.1f}%' for g in gain_b1)})")
if (gain_b1 > 0).all():
    print("-> B1 beats B0 in every fold: neighbourhood and room type carry real signal.")
else:
    print("-> B1 does not beat B0 consistently. Check price parsing and outliers, and whether many "
          "neighbourhood x room-type groups are too small for stable medians:")
    sizes = train_df.groupby(BASELINE_GROUPS, observed=True).size()
    print(f"   {len(sizes)} groups; {(sizes < B1_MIN_COUNT).sum()} have fewer than "
          f"{B1_MIN_COUNT} training rows (B1 backs off for those).")

# How much does the careful B1 differ from the simple one? (fold by fold, training CV)
b1s = cv_results["B1_simple"]
print(f"B1 (hierarchical) vs B1_simple: MAE {b1.mean():.2f} vs {b1s.mean():.2f} "
      f"({100 * (b1.mean() / b1s.mean() - 1):+.1f}%); B1 better in {(b1 < b1s).sum()}/{len(b1)} folds")

#### Check B: is the gain over B1 stable across folds?

The charter target is a **10% MAE reduction vs B1**. Here each model is compared with B1 fold by fold. This is a development check; the official number comes from `05_model_evaluation`.

In [ ]:
stability = []
for name in models:
    gain = 100 * (1 - cv_results[name] / b1)
    stability.append({
        "model": name,
        "mean reduction vs B1 (%)": gain.mean(),
        "min fold (%)": gain.min(),
        "max fold (%)": gain.max(),
        "folds beating B1": f"{(gain > 0).sum()}/{N_FOLDS}",
        "folds meeting 10%": f"{(gain >= 10).sum()}/{N_FOLDS}",
    })

stability_df = pd.DataFrame(stability).set_index("model").round(1)
stability_df

#### Check C: shuffled-price test

Prices are shuffled **between listings** in the training data (all snapshots of a listing keep the same shuffled price), which breaks any real link between features and price. A model trained on these prices can only learn noise, so it should score about the same as B0.

In [ ]:
SHUFFLE_MODEL = "hist_gbm"
rng = np.random.default_rng(RANDOM_SEED)

# Shuffle at listing level: give each listing another listing's price
listing_prices = train_df.groupby(ID_COL)[TARGET].first()
shuffled_map = pd.Series(rng.permutation(listing_prices.to_numpy()), index=listing_prices.index)
y_shuffled = train_df[ID_COL].map(shuffled_map).set_axis(y_train.index)

make_shuffle_model, log_shuffle = CV_APPROACHES[SHUFFLE_MODEL]
shuffled_mae = cv_mae(make_shuffle_model, log_shuffle, X_train, y_shuffled, cv_folds)
b0_shuffled = cv_mae(lambda: GlobalMedianBaseline(), False, X_train, y_shuffled, cv_folds)

signal_check = pd.DataFrame({
    "CV MAE": {
        f"{SHUFFLE_MODEL}, real prices": cv_results[SHUFFLE_MODEL].mean(),
        f"{SHUFFLE_MODEL}, shuffled prices": shuffled_mae.mean(),
        "B0 on shuffled prices (reference)": b0_shuffled.mean(),
    }
}).round(2)
display(signal_check)

real_gain = 100 * (1 - cv_results[SHUFFLE_MODEL].mean() / shuffled_mae.mean())
noise_vs_b0 = 100 * (shuffled_mae.mean() / b0_shuffled.mean() - 1)
print(f"Real prices give {real_gain:.1f}% lower MAE than shuffled prices.")
print(f"On shuffled prices the model is {noise_vs_b0:+.1f}% vs B0 (expected: about 0% or worse).")
if noise_vs_b0 < -5:
    print("-> Warning: the model beats B0 on shuffled prices. Check for leakage "
          "(a feature derived from price, or listings shared across folds).")

#### Reading the signal checks

- **Check A fails** (B1 not better than B0): suspect the data first, e.g. price parsing, placeholder prices, mixed currencies.
- **Check B**: a model is a credible candidate if it beats B1 in all or nearly all folds. A large spread between folds means results depend on which listings land where, so treat the mean with caution.
- **Check C fails** (model beats B0 on shuffled prices): investigate leakage before trusting any result.

With synthetic data these checks only show the code works, because the synthetic prices were built with a known pattern.

### 6.2 G2: tuned gradient boosting with an out-of-fold kNN price feature (design §6, §7.1, §8)

The top rung of the model ladder adds two things to G1:

* **kNN price:** the median price of the *k* nearest training listings, the strongest local-market signal. It is built **inside the pipeline**, so in every cross-validation fold it is computed from that fold's training rows only, and a listing's own price never enters its own value (`src/models/knn_features.py`, tested in `tests/test_modeling_extras.py`).
* **Tuning:** a seeded random search over the boosting settings and *k*, with `N_TRIALS` configurations, scored by MAE in currency on the same grouped folds as every other rung, with early stopping inside each fit. The design suggests Optuna; random search avoids a new dependency and keeps the design's other requirements (`src/models/tuning.py`).

**Watch out:** choosing the best of `N_TRIALS` configurations by their CV score makes that score optimistic. The honest comparison is on the untouched test partition in `05_model_evaluation`.

In [ ]:
def build_g2(params):
    """G2 pipeline for a parameter set: boosting settings + kNN k."""
    prep = make_preprocessor(NUMERIC_FEATURES, NUMERIC_INDICATOR_FEATURES, CATEGORICAL_FEATURES,
                             knn_k=params["knn_k"])
    return make_model("hist_gbm_tuned", prep, seed=RANDOM_SEED,
                      params={k: v for k, v in params.items() if k != "knn_k"})


if TUNE_G2:
    g2_trials = random_search(build_g2, X_train, y_train, cv_folds, to_model_target,
                              from_model_target, n_trials=N_TRIALS, seed=RANDOM_SEED)
    G2_PARAMS = best_params(g2_trials)
else:
    g2_trials = None
    G2_PARAMS = {**DEFAULT_G2_PARAMS, "knn_k": DEFAULT_KNN_K}
print(f"G2 settings: {G2_PARAMS}")

# G2 joins the ladder: fitted on the full training partition, scored on the same folds
models["hist_gbm_tuned"] = build_g2(G2_PARAMS).fit(X_train, to_model_target(y_train))
CV_APPROACHES["hist_gbm_tuned"] = (lambda: build_g2(G2_PARAMS), True)
cv_scores["hist_gbm_tuned"], cv_oof["hist_gbm_tuned"] = cv_evaluate(
    lambda: build_g2(G2_PARAMS), True, X_train, y_train, cv_folds)

# What does the kNN feature itself add? Same settings, without it (development check only)
no_knn = lambda: build_g2({**G2_PARAMS, "knn_k": None})
cv_scores["hist_gbm_tuned_no_knn"], cv_oof["hist_gbm_tuned_no_knn"] = cv_evaluate(
    no_knn, True, X_train, y_train, cv_folds)
for n in ["hist_gbm_tuned", "hist_gbm_tuned_no_knn"]:
    cv_results[n] = cv_scores[n]["MAE"]
print(f"CV MAE: G2 {cv_results['hist_gbm_tuned'].mean():.2f}, G2 without kNN "
      f"{cv_results['hist_gbm_tuned_no_knn'].mean():.2f}, G1 {cv_results['hist_gbm'].mean():.2f} "
      f"(G2's score is optimistic: it was selected on these folds)")

In [ ]:
# Trial history (appendix material) and a rough view of which settings mattered
if g2_trials is not None:
    import matplotlib.pyplot as plt
    REPORTS_DIR = REPO_ROOT / "reports" / ("synthetic_demo" if USE_SYNTHETIC else CITY_SLUG)
    REPORTS_DIR.mkdir(parents=True, exist_ok=True)
    g2_trials.to_csv(REPORTS_DIR / "04_g2_tuning_trials.csv")

    fig, ax = plt.subplots(figsize=(8, 3.5))
    ax.plot(g2_trials.index + 1, g2_trials["MAE mean"], "o", alpha=0.6, label="trial")
    ax.plot(g2_trials.index + 1, g2_trials["MAE mean"].cummin(), "-", label="best so far")
    ax.axhline(cv_results["hist_gbm"].mean(), color="gray", ls="--", label="G1 (default)")
    ax.set(xlabel="trial", ylabel=f"CV MAE ({CURRENCY}/night)", title="G2 search history")
    ax.legend()
    plt.show()
    print(f"trial history saved to {(REPORTS_DIR / '04_g2_tuning_trials.csv').relative_to(REPO_ROOT)}")
    display(rough_importance(g2_trials).round(3))
    g2_trials.sort_values("MAE mean").head(10).round(4)

## 7. CV comparison (blueprint §10)

All rungs on the same grouped folds of the **training partition**: mean and standard deviation of fold MAE, RMSE and MedAE, and **paired fold differences versus B1**. Pairing matters: each model is compared with B1 on exactly the same listings in each fold, so fold-to-fold noise largely cancels.

This table decides which rungs go to `05_model_evaluation`. The charter's 10% target is shown as a development indicator only; the official number comes from the test partition in `05_model_evaluation`.

RMSE here uses the plain `exp` back-transform, which targets the median. The next cell adds the smearing-corrected RMSE for the log-target models (design 7.3). The smearing factor is saved with each model, and `05_model_evaluation` uses it for its RMSE.

`hist_gbm_tuned` is G2. `hist_gbm_tuned_no_knn` is the same model without the kNN feature: a development check of what the feature adds. It is not a candidate and is not saved.

In [ ]:
METRICS = ["MAE", "RMSE", "MedAE"]
b1_mae = cv_scores["B1_group_median"]["MAE"]

rows = []
for name, s in cv_scores.items():
    row = {"approach": name}
    for m in METRICS:
        row[f"{m} mean"] = s[m].mean()
        row[f"{m} std"] = s[m].std(ddof=1)
    diff = s["MAE"] - b1_mae                   # negative = better than B1 on that fold
    row["MAE diff vs B1 (mean)"] = diff.mean()
    row["MAE diff vs B1 (std)"] = diff.std(ddof=1)
    row["folds better than B1"] = f"{(diff < 0).sum()}/{N_FOLDS}"
    row["reduction vs B1 (%)"] = 100 * (1 - s["MAE"].mean() / b1_mae.mean())
    rows.append(row)

cv_comparison = pd.DataFrame(rows).set_index("approach")
cv_comparison["10% target (dev)"] = cv_comparison["reduction vs B1 (%)"] >= 10
print(f"Grouped {N_FOLDS}-fold CV on the training partition ({CURRENCY}/night)")
cv_comparison.round(2)

#### Smearing-corrected RMSE (Duan 1983)
A log-target model back-transformed with `exp` predicts the conditional **median**, which is the right target for MAE. RMSE rewards the conditional **mean**, which is higher for right-skewed prices.

Duan's smearing factor `mean(exp(residual))` scales the prediction up to an estimate of the mean. The residuals here are the out-of-fold log residuals from the CV above, so the factor is estimated without any listing scoring itself.

Strictly, the factor should be estimated inside each fold; using all out-of-fold residuals at once is the usual approximation and changes it very little. MAE is not affected: it keeps the plain back-transform.

In [ ]:
smearing = {}
rows = []
for name in models:
    log_resid = to_model_target(y_train).to_numpy() - to_model_target(cv_oof[name])
    smearing[name] = float(np.mean(np.exp(log_resid)))
    oof_mean = (cv_oof[name] + 1) * smearing[name] - 1           # log1p scale: (1 + y) is smeared
    rmse_folds = [np.sqrt(mean_squared_error(y_train.iloc[va], oof_mean[va])) for _, va in cv_folds]
    rows.append({"approach": name, "smearing factor": smearing[name],
                 "RMSE plain exp (mean)": cv_comparison.loc[name, "RMSE mean"],
                 "RMSE smearing-corrected (mean)": np.mean(rmse_folds),
                 "RMSE smearing-corrected (std)": np.std(rmse_folds, ddof=1)})
cv_comparison["RMSE smeared mean"] = pd.Series({r["approach"]: r["RMSE smearing-corrected (mean)"]
                                                for r in rows})
pd.DataFrame(rows).set_index("approach").round(3)

## 8. Development error checks (out-of-fold, training partition)

Where does the model fail: which neighbourhoods, room types or price ranges? These checks use **out-of-fold predictions** from the CV above, so every training listing is scored by a model that did not see it, and the test partition stays untouched.

The candidate is the learned model with the lowest mean CV MAE. This is a development pick; the formal selection rule (design §13.2) is applied in `05_model_evaluation`. The official segment analysis on test is design §16.1.

In [ ]:
best_name = min(models, key=lambda n: cv_results[n].mean())
best_model = models[best_name]
print(f"development candidate: {best_name} (mean CV MAE {cv_results[best_name].mean():.2f})")

oof_df = train_df[[ID_COL, SNAPSHOT_COL, TARGET] + CORE_CATEGORICAL].copy()
oof_df["pred_oof"] = cv_oof[best_name]
oof_df["pred_oof_b1"] = cv_oof["B1_group_median"]
oof_df["abs_error"] = (oof_df[TARGET] - oof_df["pred_oof"]).abs()
oof_df["abs_error_b1"] = (oof_df[TARGET] - oof_df["pred_oof_b1"]).abs()
assert oof_df["pred_oof"].notna().all(), "every training row should have an out-of-fold prediction"


def segment_errors(by):
    g = oof_df.groupby(by, observed=True)
    out = pd.DataFrame({
        "rows": g.size(),
        "listings": g[ID_COL].nunique(),
        f"MAE {best_name}": g["abs_error"].mean(),
        "MAE B1": g["abs_error_b1"].mean(),
    })
    out["reduction vs B1 (%)"] = 100 * (1 - out[f"MAE {best_name}"] / out["MAE B1"])
    out["small segment (<30 listings)"] = out["listings"] < 30
    return out.round(2)


segment_errors("room_type")

In [ ]:
segment_errors("neighbourhood_cleansed").sort_values(f"MAE {best_name}", ascending=False).head(15)

In [ ]:
# Error by observed-price decile (deciles from the training partition)
oof_df["price_decile"] = pd.qcut(oof_df[TARGET], 10, labels=[f"D{i}" for i in range(1, 11)])
segment_errors("price_decile")

In [ ]:
# TODO: residual vs actual plot, learning curve (blueprint §8), feature importance (§12)
# (save figures to reports/figures/)

## 9. Prediction intervals: conformalized quantile regression (blueprint §11, design 9)

Each prediction gets an **80% interval**, the investor-facing "plausible range":
1. Fit two quantile models (10th and 90th percentile, gradient boosting with quantile loss, in log space) on the **training** partition.
2. On the **calibration** partition, with one row per listing, measure how far each observed price falls outside its interval. That distance is the conformity score.
3. Widen (or narrow) every interval by `Q`, the ⌈(1 − α)(n + 1)⌉ / n quantile of those scores. This gives about 80% coverage on new listings from the same market, without trusting the quantile models to be well calibrated.

The calibration partition is used here and nowhere else, and never for tuning. Coverage is evaluated on the test partition in `05_model_evaluation` (hypothesis H3).

In [ ]:
ALPHA = 0.2     # 80% intervals

# One row per listing in calibration (exchangeability, design 9.1): its latest snapshot
cal_one = calib_df.sort_values(SNAPSHOT_COL).groupby(ID_COL).tail(1)


def quantile_model(q):
    pipe = make_model("hist_gbm", preprocessor, seed=RANDOM_SEED)
    return pipe.set_params(model=HistGradientBoostingRegressor(
        loss="quantile", quantile=q, random_state=RANDOM_SEED))


cqr = fit_cqr(quantile_model(ALPHA / 2), quantile_model(1 - ALPHA / 2),
              X_train, y_train, cal_one[FEATURES], cal_one[TARGET], alpha=ALPHA)

# Sanity checks only (not results): coverage on calibration is ~80% by construction; on train
# it is higher because the quantile models have seen those listings.
for label, frame in [("calibration", cal_one), ("train (in-sample)", train_df)]:
    lo, hi = predict_interval(cqr, frame[FEATURES])
    y = frame[TARGET].to_numpy()
    print(f"{label:18s} coverage {((y >= lo) & (y <= hi)).mean():.1%}   "
          f"median width {np.median(hi - lo):,.0f} {CURRENCY}/night")
print(f"Q = {cqr['q']:+.4f} on the log scale (interval scaled by {np.exp(cqr['q']):.3f}), "
      f"n_calibration = {cqr['n_calibration']}")

## 10. Refit and save (blueprint §13)

The models in section 6 are fitted on the **full training partition** after the target policy (calibration and test excluded), as the blueprint asks. G2 uses the settings chosen by the search in 6.2.

**Every candidate is saved, not only the development pick.** The final choice is made by the pre-registered selection rule on the test partition in `05_model_evaluation`, so 04 needs all of them.

Saved to `models/` (excluded from Git):
* `<model>_<city>_<version>.joblib` with metadata `<model>_<city>_<version>.json`, which records:
  * the split version and partition sizes;
  * the features;
  * the CV results;
  * the smearing factor;
  * the SHA-256 fingerprint of this city's data and the hash of the artifact.
* `cqr_<city>_<version>.joblib`, the interval model, tagged with the city, split version and features.

Notebook 04 recomputes these hashes and stops or warns if anything changed. **No test predictions are produced here.**

In [ ]:
import joblib

MODELS_DIR.mkdir(parents=True, exist_ok=True)
version = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M")

common = {
    "version": version,
    "synthetic_data": USE_SYNTHETIC,
    "city": CITY,
    "city_slug": CITY_SLUG,
    "currency": CURRENCY,
    "snapshot_dates": sorted(df[SNAPSHOT_COL].dt.strftime("%Y-%m-%d").unique().tolist()),
    "data_source": DATA_SOURCE,
    "data_sha256": DATA_SHA256,
    "split_file": split_name,
    "split_version": SPLIT_VERSION,
    "partitions": {"train_rows": len(train_df), "train_listings": int(train_df[ID_COL].nunique()),
                   "calibration_rows": len(calib_df),
                   "calibration_listings": int(calib_df[ID_COL].nunique()),
                   "removed_by_target_policy": {k: v.to_dict() for k, v in removed.items()},
                   "test": "excluded (evaluated in 05_model_evaluation)"},
    "fitted_on": "full training partition, after the target policy",
    "target_policy": target_policy,
    "target": TARGET,
    "target_unit": f"{CURRENCY}/night",
    "log_target": USE_LOG_TARGET,
    "features": FEATURES,
    "optional_features_not_in_dictionary": used_optional,
    "feature_audit": {"file": FEATURE_AUDIT_FILE.name, "strict": STRICT_AUDIT,
                      "numeric_with_missing_indicator": NUMERIC_INDICATOR_FEATURES,
                      "blocked_present": fs.blocked_present, "unaudited": fs.unaudited},
    "random_seed": RANDOM_SEED,
    "baseline_b1": {"definition": B1_DESCRIPTION, "min_count": B1_MIN_COUNT},
    "development_pick": best_name,
    "cv": {"n_folds": N_FOLDS, "grouped_by": ID_COL, "partition": "train",
           "results": cv_comparison.reset_index().to_dict(orient="records")},
}

for name, pipe in models.items():
    model_version = f"{name}_{CITY_SLUG}_{version}"
    artifact_path = MODELS_DIR / f"{model_version}.joblib"
    joblib.dump(pipe, artifact_path)
    metadata = {"model": name, **common,
                "hyperparameters": G2_PARAMS if name == "hist_gbm_tuned" else None,
                "knn_k": knn_k_of(pipe),
                "tuning": ({"method": "seeded random search", "n_trials": N_TRIALS,
                            "seed": RANDOM_SEED, "space": {k: list(v[:3]) for k, v in SEARCH_SPACE.items()}}
                           if name == "hist_gbm_tuned" and TUNE_G2 else None),
                "smearing_factor": smearing[name] if USE_LOG_TARGET else 1.0,
                "artifact_sha256": sha256_file(artifact_path)}
    (MODELS_DIR / f"{model_version}.json").write_text(json.dumps(metadata, indent=2, default=str))
    print(f"saved {artifact_path.name} + metadata")

cqr_path = MODELS_DIR / f"cqr_{CITY_SLUG}_{version}.joblib"
cqr.update(split_version=SPLIT_VERSION, features=FEATURES, data_sha256=DATA_SHA256,
           city=CITY, version=version)
save_cqr(cqr, cqr_path)
print(f"saved {cqr_path.name} (alpha {cqr['alpha']}, Q {cqr['q']:+.4f})")
print(f"{CITY}: split {SPLIT_VERSION}, data {DATA_SHA256[:12]}; development pick: {best_name}")

## 11. Open decisions and TODOs

- [x] Frozen three-way grouped split created by `src/models/make_split.py`; this notebook only loads and verifies it
- [x] Test partition removed from this notebook (comparison, error checks and predictions now use CV / out-of-fold on train)
- [x] Switched to the real file: `listings_master.parquet`, Amsterdam, EUR (one snapshot, scraped 15–17 June 2026)
- [x] Reads `listings_features.parquet` (03_Feature_Engineering); its 29 engineered columns are audited: competition, amenities and neighbourhood counts included, price-derived columns excluded (leakage)
- [ ] Data owner: count recent reviews up to each city's snapshot date, not the latest review in the file; then move `reviews_last_*` from defer to ablation in the audit
- [x] `split_v1_amsterdam.json` created by `src/models/make_split.py` and committed
- [x] Multi-city ready: per-city split, currency lookup, city data fingerprint and city-tagged artifacts (`src/models/cities.py`)
- [ ] Team: choose the cities to model; for each, run `make_split --city <city>` once, then this notebook and 04
- [ ] Data owner: add `snapshot_date` in `01_Data_Preparation` (derived here from `scrape_id` by `ensure_snapshot_date`)
- [ ] Data owner: about 19% of listings have no `price`. Confirm this is missing in the source and not lost in processing
- [ ] Forward test (design §4.1, §17.1) needs a later snapshot of the same city
- [ ] Agree the column names with the data owner (contract uses `listing_id` / `neighbourhood`; dictionary and this notebook use `id` / `neighbourhood_cleansed`), then add `check_contract` in `src/models/contract.py` (design §3.1)
- [x] Target policy with thresholds from the training partition only (design §3.2); rows removed per partition logged and saved
- [x] Feature list loaded from `evaluation/feature_audit.csv` via `src/models/feature_audit.py` (design §3.3)
- [x] Audit completed against the 2026-06 schema (all 93 columns decided) and `STRICT_AUDIT = True`. Review the `defer` / `ablation` choices with the team
- [ ] Amenities: parse to curated flags and a count, then change its audit decision from `planned` to `include` (design §7.3)
- [ ] Ablations for the audit's caveats: availability (excluded), reviews, minimum_nights/instant_bookable, free text (design §7.4, §17.2)
- [x] B1 is the hierarchical median (min group size 10, room-type fallback; design §6.1); the earlier version is kept as `B1_simple`
- [ ] Team: confirm the B1 definition in the README evaluation table before the real run of 05
- [x] Target distribution shown after the split, on the training partition only (blueprint §4)
- [x] Smearing-corrected RMSE for log-target models (design 7.3); factor saved for `05_model_evaluation`
- [ ] With the data owner: add every used feature with `in_dictionary = no` in the audit to `data_dictionary.md` (then flip it to `yes`), or drop it
- [x] `MedianGroupBaseline`, the parsers, the synthetic generator and the pipeline builders moved to `src/models/` (shared with `05_model_evaluation`)
- [ ] Choose and pin modelling libraries in `requirements.txt`
- [ ] Review the signal checks (section 6.1) on the first real snapshot and record the findings
- [x] G2: seeded random search with grouped CV (reuse `cv_folds`) + out-of-fold kNN price feature (design §7.1, §8)
- [ ] Optional: Optuna TPE instead of random search, if the team adds the dependency
- [x] Quantile models and conformal calibration on the calibration partition (blueprint §11); `cqr_<version>.joblib` saved
- [x] Every candidate saved with data/artifact hashes, so `05_model_evaluation` evaluates exactly what was trained here
- [ ] Notebook 04: commit the pre-registration, then run it once on the real test partition